In [75]:
import pandas as pd
import re

raw = pd.read_csv('/content/drive/MyDrive/california_restaurants.csv')
df = raw.copy()
print(df.shape)
df.head()

(1273, 18)


,Yelp URL,Name,Street Address,Zip Code,City,State,Price Range,Phone,Rating,Number of Reviews,Website,Menu Link,Image 1,Image 2,Image 3,Category 1,Category 2,Category 3
0,https://www.yelp.com/biz/in-s%C4%ABt-coffee-bu...,In-sīt Coffee,6930 Beach Blvd Ste L301,90621.0,Buena Park,CA,$,(714) 670-6958,4.0,583,NaN,NaN,https://s3-media0.fl.yelpcdn.com/bphoto/TgX8pH...,https://s3-media0.fl.yelpcdn.com/bphoto/e9oSgM...,https://s3-media0.fl.yelpcdn.com/bphoto/gJUxfa...,Coffee & Tea,Waffles,Breakfast & Brunch
1,https://www.yelp.com/biz/tierra-mia-coffee-wes...,Tierra Mia Coffee,2301 S Azusa Ave,91792.0,West Covina,CA,$$,NaN,4.0,288,www.tierramiacoffee.com,NaN,https://s3-media0.fl.yelpcdn.com/bphoto/qQC8B3...,https://s3-media0.fl.yelpcdn.com/bphoto/QhVb_D...,https://s3-media0.fl.yelpcdn.com/bphoto/7RiGO2...,Coffee & Tea,NaN,NaN
2,https://www.yelp.com/biz/julies-cafe-chino-3,Julie's Cafe,3746 Riverside Dr,91710.0,Chino,CA,$$,(909) 992-0013,4.5,179,www.juliescafes.com,www.juliescafes.com/menu/,https://s3-media0.fl.yelpcdn.com/bphoto/__RSTK...,https://s3-media0.fl.yelpcdn.com/bphoto/bdPEPF...,https://s3-media0.fl.yelpcdn.com/bphoto/7d4bRB...,Bagels,Breakfast & Brunch,Cafes
3,https://www.yelp.com/biz/bonanza-bakery-and-ca...,Bonanza Bakery & Cafe,"20657 Golden Springs Dr Ste - 104,105",91789.0,Diamond Bar,CA,$$,(909) 274-7121,4.0,76,www.bonanzabakerycafe.com,NaN,https://s3-media0.fl.yelpcdn.com/bphoto/ca8BLo...,https://s3-media0.fl.yelpcdn.com/bphoto/FVqd4a...,https://s3-media0.fl.yelpcdn.com/bphoto/3iL4th...,Desserts,Coffee & Tea,Breakfast & Brunch
4,https://www.yelp.com/biz/dripp-fullerton,Dripp,,NaN,500 N Harbor Blvd,NaN,$,(714) 441-1003,4.0,938,www.dripp.com,NaN,https://s3-media0.fl.yelpcdn.com/bphoto/zdCaor...,https://s3-media0.fl.yelpcdn.com/bphoto/Tpbn2s...,https://s3-media0.fl.yelpcdn.com/bphoto/nXoE1M...,Coffee & Tea,Ice Cream & Frozen Yogurt,Sandwiches


In [76]:
# Delete rows with missing/blank values
df = df.dropna(subset=['Name', 'Street Address', 'City', 'State', 'Zip Code',
                        'Rating', 'Number of Reviews', 'Phone', 'Price Range'])

# At least one category must exist
df = df[df[['Category 1', 'Category 2', 'Category 3']].notna().any(axis=1)]

# Drop rows with placeholder values
df = df[df['Price Range'] != 'Unknown']
df = df[df['Phone'].str.strip() != '']

print(f"After missing value removal: {len(df)} rows")
print(df.isnull().sum())

After missing value removal: 1017 rows
Yelp URL               0
Name                   0
Street Address         0
Zip Code               0
City                   0
State                  0
Price Range            0
Phone                  0
Rating                 0
Number of Reviews      0
Website              189
Menu Link            501
Image 1                0
Image 2                2
Image 3                3
Category 1             0
Category 2           179
Category 3           438
dtype: int64


In [77]:
#drop duplicates
df = df.drop_duplicates(subset=['Yelp URL'])
print(f"After deduction: {len(df)} rows")

After deduction: 660 rows


In [78]:
#Fixing data types/output
#Zip
df['Zip Code'] = df['Zip Code'].apply(lambda z: str(int(float(z))).zfill(5) if pd.notna(z) else None)

#Rating
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')

print(df[['Zip Code', 'Rating']].dtypes)

Zip Code     object
Rating      float64
dtype: object


In [79]:
#Fixing typo/formatting
#State
df['State'] = df['State'].str.strip().str.upper()

#Price range
df['Price Range'] = df['Price Range'].where(df['Price Range'].isin(['$', '$$', '$$$', '$$$$']))

#Clean all the gaps and whitespaces
for col in ['Name', 'Street Address', 'City', 'Phone', 'Website', 'Category 1', 'Category 2', 'Category 3']:
    df[col] = df[col].str.strip()

#Phone Formatting
def clean_phone(p):
    if pd.isna(p): return None
    digits = re.sub(r'\D', '', str(p))
    if digits.startswith('1') and len(digits) == 11:
        digits = digits[1:]
    return f'({digits[:3]}) {digits[3:6]} {digits[6:]}' if len(digits) == 10 else None

df['Phone'] = df['Phone'].apply(clean_phone)

print("Formats standardised!")
print(df[['State', 'Price Range', 'Phone']].head())

Formats standardised!
  State Price Range           Phone
0    CA           $  (714) 670 6958
2    CA          $$  (909) 992 0013
3    CA          $$  (909) 274 7121
5    CA          $$  (909) 551 0021
6    CA          $$  (626) 581 1581


In [80]:
# Finding outliers
#Valid Rating
print(f"Invalid ratings: {df[~df['Rating'].between(1, 5)].shape[0]}")
df = df[df['Rating'].between(1, 5)]

#Drop rows with 0 reviews
df['Number of Reviews'] = pd.to_numeric(df['Number of Reviews'], errors='coerce')
df = df[df['Number of Reviews'] > 0]

#Find outliers
df['review_outliers'] = df['Number of Reviews'] > 5000
print(f"Outliers: {df['review_outliers'].sum()}")

print(f"\nRemaining rows: {len(df)} rows")

Invalid ratings: 0
Outliers: 2

Remaining rows: 659 rows


In [81]:
print("Cleansing update")
print(f"Raw rows:     {len(raw)}")
print(f"Clean rows:   {len(df)}")
print(f"Rows removed: {len(raw) - len(df)}")

print("\nOptional Fields that have remaining nulls")
print(df.isnull().sum())

Cleansing update
Raw rows:     1273
Clean rows:   659
Rows removed: 614

Optional Fields that have remaining nulls
Yelp URL               0
Name                   0
Street Address         0
Zip Code               0
City                   0
State                  0
Price Range            0
Phone                  0
Rating                 0
Number of Reviews      0
Website              137
Menu Link            334
Image 1                0
Image 2                1
Image 3                2
Category 1             0
Category 2           127
Category 3           296
review_outliers        0
dtype: int64


In [82]:
from google.colab import files
df.to_csv('california_restaurants_cleaned.csv', index=False)
files.download('california_restaurants_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>